In [238]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0

In [239]:
from google.colab import drive
import sys

In [240]:
drive.mount('/content/drive')
sys.path.append(
"/content/drive/MyDrive/SupportAI"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [241]:
import numpy as np
from datasets import load_from_disk
import matplotlib.pyplot as plt
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import json
from pathlib import Path
from datasets import Value

In [255]:
feature = "topic"

path = '/content/drive/MyDrive/SupportAI'
with open(f'{path}/label_mappings.json', 'r') as f:
    label_mapping = json.load(f)

id2label = {int(k): v for k, v in label_mapping[feature]['id2label'].items()}
label2id = {k: int(v) for k, v in label_mapping[feature]['label2id'].items()}
label_mapping.keys()

dict_keys(['topic', 'intent', 'sentiment', 'urgency'])

In [256]:
train_dataset = load_from_disk(f"{path}/data/processed/train")
val_dataset = load_from_disk(f"{path}/data/processed/val")
test_dataset = load_from_disk(f"{path}/data/processed/test")

In [257]:
train_dataset = train_dataset.rename_column(
    f"{feature}_label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    f"{feature}_label",
    "labels"
)

test_dataset = test_dataset.rename_column(
    f"{feature}_label",
    "labels"
)

In [258]:
train_dataset.features["labels"]

Value('int64')

In [259]:
model_name = 'DeepPavlov/rubert-base-cased'
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id, problem_type="single_label_classification")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those par

In [260]:
print(model.config.problem_type)
print(model.config.num_labels)

single_label_classification
7


In [261]:
print(train_dataset.features["labels"])

Value('int64')


### Функция оценки метрик (есть смысл перенести в отдельный файл)

In [262]:
def metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [263]:
train_dataset.features

{'Unnamed: 0': Value('int64'),
 'dialogue_id': Value('int64'),
 'turn': Value('int64'),
 'text': Value('string'),
 'role': Value('string'),
 'topic': Value('string'),
 'intent': Value('string'),
 'sentiment': Value('string'),
 'urgency': Value('int64'),
 'labels': Value('int64'),
 'intent_label': Value('int64'),
 'sentiment_label': Value('int64'),
 'urgency_label': Value('float64'),
 '__index_level_0__': Value('int64'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

#### Настройки обучения

In [264]:
training_args = TrainingArguments(
    output_dir="models/topic",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [265]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=metrics,
)

In [266]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.324333,0.322708,0.879121,0.882970,0.897186,0.879121
2,0.288183,0.280091,0.885400,0.895217,0.929364,0.885400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
test_results = trainer.evaluate(
    test_dataset
)

test_results

In [ ]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)

labels = predictions.label_ids

In [ ]:
class_names = list(model.config.id2label.values())

### Аналогично, есть смысл перенести в отдельный файл

In [ ]:
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    values_format="d",
    colorbar=False,
    ax=ax
)

ax.set_title("Confusion Matrix", fontsize=16, pad=15)
ax.set_xlabel("Predicted label", fontsize=13)
ax.set_ylabel("True label", fontsize=13)

plt.xticks(fontsize=11, rotation=45)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
save_dir = Path(f"/content/drive/MyDrive/SupportAI/models/{feature}")

trainer.save_model(save_dir)